# Lung atlas (K=13, fine-grained cell types) -- paper-exact comparison

Reproduces the arXiv:2607.25031 Table 1 comparison exactly: the original 7 methods, `pooled_concat` (the raw-data-concatenation pooled estimator) filling the "Pooled" slot.

**Method set (7 methods, "Pooled" = `pooled_concat`):** `target_only`,
`multi_source_pooled`, `pooled_concat` (displayed as **"Pooled"**),
`adaptive_multi_source`, `tlgmm`, `scrna`, `gdec_gcnfree`. `target_source_pooled` is
excluded entirely from this notebook -- it's the newer target+source pooled-subspace estimator, not part of the published comparison -- see `aggregate_and_plot_new_pooled.ipynb` for that version. Per
project convention, the two pooled variants (`pooled_concat` and
`target_source_pooled`) never appear together in the same table/plot/tex file.

Reads `results_0.5_alpha/raw/*.csv` (written by `run_lung_atlas_comparison.py`),
filters to the 7 methods above, and reproduces the standard comparison: a
combined ARI/V-measure/misclustering LaTeX table and two bar charts
(misclustering error, ARI). (For the one-source-at-a-time sensitivity analysis
specifically for `multi_source_pooled`, see the separate
`single_source_sensitivity.ipynb` -- not part of this notebook.)

Standalone and read-only with respect to `results_0.5_alpha/raw/` -- rerun any
time after adding/replacing files there. Outputs land in `results_0.5_alpha/paper/`:
`lung_atlas_comparison_results.csv`, `lung_atlas_combined_table.tex`,
`lung_atlas_misclustering.pdf`, `lung_atlas_ari.pdf`.

In [ ]:
import glob, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common

## Load, filter to the 7-method set, and combine

Reads every `results_0.5_alpha/raw/*.csv`, keeps only rows for the 7 methods
this notebook uses (`target_source_pooled` excluded), and checks completeness against
the expected grid restricted to those 7 methods before doing anything else.

In [ ]:
RESULTS_DIR = "results_0.5_alpha"
RAW_DIR = os.path.join(RESULTS_DIR, "raw")
OUT_DIR = "results_0.5_alpha/paper"
os.makedirs(OUT_DIR, exist_ok=True)

raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(raw_paths)} raw result files in {RAW_DIR}")
assert raw_paths, f"No CSVs found in {RAW_DIR}"

# This notebook's 7-method set -- "target_source_pooled" is deliberately excluded (see
# intro markdown); the pooled slot is filled by "pooled_concat", always
# displayed as "Pooled". Never show pooled_concat and target_source_pooled
# together in the same table/plot/tex file (project convention).
INCLUDED_METHODS = [
    "target_only", "multi_source_pooled", "pooled_concat",
    "adaptive_multi_source", "tlgmm", "scrna", "gdec_gcnfree",
]
LABELS = dict(common.METHOD_LABELS)
LABELS["pooled_concat"] = "Pooled"

# The methods (of the 7 above) that take a `sources` list and get both an
# "all" run and one single-source run per other batch -- mirrors
# run_lung_atlas_comparison.py's MULTI_SOURCE_METHODS, restricted to this
# notebook's method set.
MULTI_SOURCE_METHODS = {m for m in
    ["multi_source_pooled", "pooled_concat", "target_source_pooled", "adaptive_multi_source"]
    if m in INCLUDED_METHODS}


def expected_grid():
    """Mirrors run_lung_atlas_comparison.py's task_grid(), restricted to
    INCLUDED_METHODS, as a set of (target, method, source) triples."""
    grid = set()
    for target in common.BATCHES:
        others = [b for b in common.BATCHES if b != target]
        for method in INCLUDED_METHODS:
            if method == "target_only":
                grid.add((target, method, "none"))
            elif method in MULTI_SOURCE_METHODS:
                grid.add((target, method, "all"))
                for src in others:
                    grid.add((target, method, src))
            else:
                grid.add((target, method, "all"))
    return grid


expected = expected_grid()
found = set()
rows = []
for path in raw_paths:
    row = pd.read_csv(path).iloc[0]
    if row["method"] not in INCLUDED_METHODS:
        continue
    rows.append(row.to_dict())
    found.add((row["target"], row["method"], row["source"]))

missing = expected - found
if missing:
    print(f"WARNING: {len(missing)}/{len(expected)} (target, method, source) results missing from {RAW_DIR}:")
    for target, method, source in sorted(missing):
        print(f"  {target}, {method}, {source}")
else:
    print(f"All {len(expected)} (target, method, source) combinations present for this notebook's method set")

results = pd.DataFrame(rows)
combined_path = os.path.join(OUT_DIR, "lung_atlas_comparison_results.csv")
results.to_csv(combined_path, index=False)
print(f"Wrote {combined_path}")

# The tables/plot below reproduce the one-row-per-(target, method) comparison,
# so they only use the all-sources-pooled (or no-source, for target_only)
# rows -- the single-source rows are used later in "Per-source sensitivity".
results_all = results[results["source"].isin(["all", "none"])].reset_index(drop=True)
results_all

## Summary tables (method x target batch)

In [ ]:
for metric in ["misclustering", "ari", "v_measure"]:
    print(f"\n--- {metric} ---")
    display(results_all.pivot(index="method", columns="target", values=metric)
                        .reindex(index=INCLUDED_METHODS, columns=common.BATCHES))

## Combined LaTeX booktabs table

One column group per target batch, each split into three subcolumns -- ARI,
V-measure, $\mathcal{L}_{\mathrm{mult}}$ (misclustering error) -- one row per
method. ARI/V-measure are better when **higher**; $\mathcal{L}_{\mathrm{mult}}$
is better when **lower**; the best value in each subcolumn is bolded. Wrapped in
`\resizebox` since 1 + 4*3 = 13 columns is wider than a standard text column.

In [ ]:
SUBCOLS = [("ari", "ARI", "high"), ("v_measure", "V-measure", "high"),
           ("misclustering", "$\\mathcal{L}_{\\mathrm{mult}}$", "low")]


def make_combined_latex_table(results: pd.DataFrame, methods, labels, caption: str = "", label: str = "") -> str:
    """Single booktabs table: rows = methods, columns = (batch, metric) for
    every batch x {ari, v_measure, misclustering} pair, best value per
    (batch, metric) subcolumn bolded (ari/v_measure: higher is better;
    misclustering: lower is better). `results` should already be filtered to
    one row per (method, target) -- e.g. `results_all` -- since `pivot`
    errors on duplicate (method, target) pairs."""
    pivots = {
        metric: results.pivot(index="method", columns="target", values=metric)
                        .reindex(index=methods, columns=common.BATCHES)
        for metric, _, _ in SUBCOLS
    }

    n_sub = len(SUBCOLS)
    lines = []
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\resizebox{\\textwidth}{!}{%")
    lines.append("\\begin{tabular}{l" + "ccc" * len(common.BATCHES) + "}")
    lines.append("\\toprule")

    header1 = [""]
    cmidrules = []
    col = 2
    for batch in common.BATCHES:
        header1.append(f"\\multicolumn{{{n_sub}}}{{c}}{{{batch.replace('_', chr(92) + '_')}}}")
        cmidrules.append(f"\\cmidrule(lr){{{col}-{col + n_sub - 1}}}")
        col += n_sub
    lines.append(" & ".join(header1) + " \\\\")
    lines.append(" ".join(cmidrules))

    header2 = ["Method"] + [sub_label for _ in common.BATCHES for _, sub_label, _ in SUBCOLS]
    lines.append(" & ".join(header2) + " \\\\")
    lines.append("\\midrule")

    for method in methods:
        cells = []
        for batch in common.BATCHES:
            for metric, _, direction in SUBCOLS:
                pivot = pivots[metric]
                val = pivot.loc[method, batch]
                best = pivot[batch].min() if direction == "low" else pivot[batch].max()
                cell = f"{val:.3f}"
                if np.isclose(val, best):
                    cell = f"\\textbf{{{cell}}}"
                cells.append(cell)
        label_str = labels[method].replace("_", "\\_")
        lines.append(f"{label_str} & " + " & ".join(cells) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}%")
    lines.append("}")
    if caption:
        lines.append(f"\\caption{{{caption}}}")
    if label:
        lines.append(f"\\label{{{label}}}")
    lines.append("\\end{table}")
    return "\n".join(lines)


caption = ("ARI, V-measure, and misclustering error ($\\mathcal{L}_{\\mathrm{mult}}$) on the lung "
           "atlas (13 fine-grained cell types, K=13), leave-one-Dropseq-batch-out. ARI/V-measure: "
           "higher is better; $\\mathcal{L}_{\\mathrm{mult}}$: lower is better. Best value per "
           "subcolumn in bold.")
combined_tex = make_combined_latex_table(results_all, INCLUDED_METHODS, LABELS, caption=caption,
                                          label="tab:lung_k13_combined")
print(combined_tex)

table_path = os.path.join(OUT_DIR, "lung_atlas_combined_table.tex")
with open(table_path, "w") as f:
    f.write(combined_tex)
    f.write("\n")
print(f"Wrote {table_path}")

## Figures: misclustering error and ARI by target batch

Grouped bar charts -- one group per target Dropseq batch, one bar per method (the
`source="all"`/`"none"` rows only, i.e. `results_all`). Colors are fixed per method
(never reassigned across methods, and the same across both charts), using this
repo's validated 7-slot categorical order (blue / aqua / magenta / yellow / green
/ violet / red) so adjacent bars stay distinguishable under color-vision
deficiency (verified with `scripts/validate_palette.js` -- ALL CHECKS PASS in this
dict order for this 7-method set). The aqua, magenta, and yellow slots are
low-contrast on a light surface, so each bar also gets a direct value label on
top.

In [ ]:
METHOD_COLORS = {
    "target_only": "#2a78d6",             # slot 1 blue
    "multi_source_pooled": "#1baf7a",     # slot 3 aqua
    "pooled_concat": "#e87ba4",        # slot 5 magenta ("Pooled")
    "adaptive_multi_source": "#eda100",   # slot 4 yellow
    "tlgmm": "#008300",                   # slot 6 green
    "scrna": "#4a3aa7",                   # slot 7 violet
    "gdec_gcnfree": "#e34948",            # slot 8 red
}


def plot_metric_bar_chart(metric: str, ylabel: str, title_suffix: str, out_name: str):
    pivot = results_all.pivot(index="method", columns="target", values=metric).reindex(
        index=INCLUDED_METHODS, columns=common.BATCHES
    )

    n_methods = len(INCLUDED_METHODS)
    x = np.arange(len(common.BATCHES))
    width = 0.8 / n_methods

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, method in enumerate(INCLUDED_METHODS):
        offset = (i - (n_methods - 1) / 2) * width
        vals = pivot.loc[method].values
        bars = ax.bar(x + offset, vals, width, label=LABELS[method], color=METHOD_COLORS[method])
        ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(common.BATCHES)
    ax.set_ylabel(ylabel)
    ax.set_title(f"Lung atlas (K=13, fine-grained cell types): {title_suffix} by target batch")
    ymin = min(0.0, pivot.values.min() * 1.1)
    ax.set_ylim(ymin, max(pivot.values.max() * 1.25, 0.05))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    fig.tight_layout()

    fig_path = os.path.join(OUT_DIR, out_name)
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Wrote {fig_path}")
    plt.show()


plot_metric_bar_chart("misclustering", "misclustering error (lower is better)",
                       "misclustering error", "lung_atlas_misclustering.pdf")
plot_metric_bar_chart("ari", "Adjusted Rand Index (higher is better)",
                       "ARI", "lung_atlas_ari.pdf")